In [ ]:
!pip install -q langchain langchain-groq langchain-text-splitters langchain-huggingface sentence-transformers

import os
from google.colab import userdata
from IPython.display import Markdown

In [ ]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("langsmith_api_key")
os.environ["LANGSMITH_PROJECT"] = "shopeasy-rag1"
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
%%writefile travel_booking_policy.txt
TRAVEL BOOKING POLICY
Version: 1.0

1. Booking support
The travel assistant can search flights and hotels, provide destination weather information, estimate trip costs, and prepare an itinerary. It does not purchase tickets, reserve rooms, or collect payment. It must clearly tell the customer that search results and estimates are not confirmed bookings.

2. Information required
For flight search, collect origin, destination, departure date, number of adults, and cabin class. For a return trip, also collect return date. Ask about children and infants when relevant.
For hotel search, collect destination, check-in date, check-out date, and number of guests/rooms. Ask for a preferred budget or hotel category if the customer has one.
For weather, collect the destination and travel dates.
For a trip budget, collect the number of travelers, trip duration, and known or estimated flight, hotel, food, local transport, and activity costs.

3. Changes and cancellations
The assistant must not claim that it can change or cancel a real booking. It may explain general next steps and direct the customer to the airline, hotel, or booking provider. Actual change and cancellation rules depend on the provider and fare/room conditions.

4. Customer protection
Never request a card number, CVV, banking password, passport scan, or one-time password. Do not expose another customer's booking details. Do not invent booking references, prices, availability, baggage allowances, or cancellation terms.

5. Recommendations
Explain the main trade-offs: price, travel duration, number of stops, location, guest rating, and cancellation conditions when that information is available. Label missing information as unknown rather than guessing.


Overwriting travel_booking_policy.txt


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Travel_app") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
def load_flights_data():
  schema='flight_id string,airline string,flight_number string,origin string,destination string,departure_datetime string,arrival_datetime string,duration_minutes int,stops int,price_inr_per_person double,cabin_class string,baggage_note string'
  flights_df=spark.read.schema(schema).option('header',True).format('csv').load('/content/sample_flights.csv')
  flights_df=flights_df.withColumn('departure_datetime',try_to_timestamp(col('departure_datetime'),lit('yyyy-MM-dd HH:mm')))
  flights_df=flights_df.withColumn('departure_date',date_format('departure_datetime','yyyy-MM-dd'))
  flights_df=flights_df.withColumn('departure_time',date_format('departure_datetime','HH:mm'))
  flights_df=flights_df.withColumn('arrival_datetime',try_to_timestamp(col('arrival_datetime'),lit('yyyy-MM-dd HH:mm')))
  flights_df=flights_df.withColumn('arrival_date',date_format('arrival_datetime','yyyy-MM-dd'))
  flights_df=flights_df.withColumn('arrival_time',date_format('arrival_datetime','HH:mm'))
  return flights_df

In [ ]:
def load_hotels_data():
  schema='hotel_id string,hotel_name string,city string,area string,nightly_price_inr int,rating double,amenties string,cancellation_note string'
  hotels_df=spark.read.format('csv').option('header',True).option('inferSchema',True).load('/content/sample_hotels.csv')
  return hotels_df

In [ ]:
from typing import Optional, Literal
from datetime import date
from langchain.tools import tool
from pyspark.sql.functions import col
@tool
def search_hotels(destination:str,check_in:date,check_out:date,guests:int,
                  rooms: Optional[int] = None,
    budget: Optional[float] = None,
    star_rating: Optional[float] = None,
    amenities: Optional[list[str]] = None,
    preferred_area: Optional[str] = None):
  """Search hotels based on destination, dates, guests, budget and preferences.
  How to present results:
Show property name, area/address, nightly price, total stay price when available, rating and review count when available, included amenities, and cancellation terms when provided.

Rules:
- Check-out must be later than check-in.
- Do not invent ratings, amenities, taxes, fees, or cancellation terms.
- Clarify whether displayed prices include taxes and fees.
- Do not describe a hotel as available unless the search source confirms availability for the exact dates and occupancy
  """
  if guests <= 0:
        return("Guests must be greater than zero.")

  if check_out <= check_in:
        return ("Check-out date must be after check-in date.")
  else:
    hotels_df=load_hotels_data()
    filters = [
        col("city") == destination or col('area')==destination]
    if budget is not None:
      filters.append(col('nightly_price_inr')<=budget)
    if star_rating is not None:
      filters.append(col('rating')<=star_rating)
    result = hotels_df.filter(
        reduce(lambda a, b: a & b, filters)
    )
    if amenities:
        result = result.filter(
            array_intersect(col("amenities"), amenities).isEmpty() == False
        )
    return [row.asDict() for row in result.limit(10).collect()]

In [ ]:
from typing import Optional, Literal
from datetime import date
from langchain.tools import tool
from pyspark.sql.functions import col
@tool
def search_flights(origin:str,destination:str,departure_date:date,adults:int,return_date:Optional[date]=None,
                   children:Optional[int]=None,infants:Optional[int]=None,cabin_class:Optional[str]=None,
                   max_stops:Optional[int]=None,preferred_airline:Optional[str]=None):
  '''Search the flights based on origin,destination,departure date,adults
  How to present results:
Show property name, area/address, nightly price, total stay price when available, rating and review count when available, included amenities, and cancellation terms when provided.

Rules:
- Check-out must be later than check-in.
- Do not invent ratings, amenities, taxes, fees, or cancellation terms.
- Clarify whether displayed prices include taxes and fees.
- Do not describe a hotel as available unless the search source confirms availability for the exact dates and occupancy.
'''
  if adults <=0:
    return 'Adults must be greater than zero'
  if return_date is not None:
    if return_date<=departure_date:
     return 'Return date must be after departure date'
  flights_df=load_flights_data()
  filters=[
      col('origin')==origin,
      col('destination')==destination,
      col('departure_date')==departure_date,
  ]
  if return_date is not None:
    if return_date<=departure_date:
     return 'Return date must be after departure date'
    else:
      filters.append(col('arrival_date')==return_date)
  if children is not None or infants is not None:
    if children<0 or infants<0:
      return 'Children and infants cannot be negative.'
  if cabin_class is not None:
    if cabin_class not in ['economy', 'premium_economy', 'business', 'first']:
       return 'Invalid cabin class.'
    else:
      filters.append(col('cabin_class')==cabin_class)
  if max_stops is not None:
        filters.append(col('stops') <= max_stops)
  if preferred_airline is not None:
    filters.append(col('airline')==preferred_airline)
  result = flights_df.filter(
        reduce(lambda a, b: a & b, filters)
    )
  return [row.asDict() for row in result.limit(10).collect()]

In [ ]:
@tool
def calculate_trip_budget(adults:int,nights:int,destination:str,
    food_per_person_per_day: float = 0,rooms: int = 1,
    local_transport: float = 0,
    activities: float = 0):
     '''Calculate the total trip budget based on flight id, hotel id, adults,nights and other data provided.
     Do not provide any false information and calculate total budget based upon data provided.
     '''
     flights_df=load_flights_data()
     flights_filter_df=flights_df.filter(col('destination')==destination)
     price_per_person=flights_filter_df.select('price_inr_per_person').collect()[0][0]
     flight_cost=price_per_person*adults
     hotels_df=load_hotels_data()
     hotels_filter_df=hotels_df.filter(col('city')==destination)
     night_cost=hotels_filter_df.select('nightly_price_inr').collect()[0][0]
     hotel_cost=night_cost*nights*rooms
     food_cost = food_per_person_per_day * adults * (nights + 1)
     total_cost=flight_cost+hotel_cost+food_cost+local_transport+activities
     per_person_budget=total_cost/adults
     return {
        "flight_cost": flight_cost,
        "hotel_cost": hotel_cost,
        "food_cost": food_cost,
        "total_budget": total_cost,
        "per_person_budget": total_cost / adults
    }

In [ ]:
%%writefile travel_booking_policy.txt
TRAVEL BOOKING POLICY
Version: 1.0

1. Booking support
The travel assistant can search flights and hotels, provide destination weather information, estimate trip costs, and prepare an itinerary. It does not purchase tickets, reserve rooms, or collect payment. It must clearly tell the customer that search results and estimates are not confirmed bookings.

2. Information required
For flight search, collect origin, destination, departure date, number of adults, and cabin class. For a return trip, also collect return date. Ask about children and infants when relevant.
For hotel search, collect destination, check-in date, check-out date, and number of guests/rooms. Ask for a preferred budget or hotel category if the customer has one.
For weather, collect the destination and travel dates.
For a trip budget, collect the number of travelers, trip duration, and known or estimated flight, hotel, food, local transport, and activity costs.

3. Changes and cancellations
The assistant must not claim that it can change or cancel a real booking. It may explain general next steps and direct the customer to the airline, hotel, or booking provider. Actual change and cancellation rules depend on the provider and fare/room conditions.

4. Customer protection
Never request a card number, CVV, banking password, passport scan, or one-time password. Do not expose another customer's booking details. Do not invent booking references, prices, availability, baggage allowances, or cancellation terms.

5. Recommendations
Explain the main trade-offs: price, travel duration, number of stops, location, guest rating, and cancellation conditions when that information is available. Label missing information as unknown rather than guessing.

Overwriting travel_booking_policy.txt


In [ ]:
%%writefile destination_kit.txt
SAMPLE DESTINATION FACTS (TRAINING DATA ONLY)

Destination: Jaipur, India
Common visitor interests: forts, palaces, markets, museums, local food, photography.
Suggested trip length for a first-time visitor: 2-3 days (guideline, not a requirement).
Useful planning note: group attractions by area to reduce cross-city travel.
Weather note: always retrieve a date-specific forecast before making weather claims.

Destination: Goa, India
Common visitor interests: beaches, heritage areas, local cuisine, water activities, markets.
Suggested trip length for a relaxed first-time visit: 3-5 days (guideline, not a requirement).
Useful planning note: travel time can vary substantially by beach area and traffic.
Weather note: always retrieve a date-specific forecast before making weather claims.

Destination: Delhi, India
Common visitor interests: historical monuments, museums, markets, food, gardens.
Suggested trip length for a first-time visitor: 2-3 days (guideline, not a requirement).
Useful planning note: group sightseeing by area and account for city traffic.
Weather note: always retrieve a date-specific forecast before making weather claims.

These are fictional training guidelines, not live travel data. Verify current access rules, prices, hours, and weather before presenting them as facts.


Overwriting destination_kit.txt


In [ ]:
%%writefile itinerary_planning.text

Overwriting itinerary_planning.text


In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
documents = []
for filename in ["destination_kit.txt", "travel_booking_policy.txt"]:

  text = open('destination_kit.txt').read()

  documents.append(
        Document(
            page_content=text,
            metadata={"source":filename}
        )
     )

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 16


In [ ]:
# 3. CREATE EMBEDDINGS
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
!pip install langchain-qdrant

In [ ]:
from langchain_qdrant import QdrantVectorStore
vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="travel_knowledge"
)

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
def search_knowledge(question: str, source: str = None):
    if source:
        results = vector_store.similarity_search(
            question,
            k=3,
            filter={
                "must": [
                    {
                        "key": "metadata.source",
                        "match": {"value": source}
                    }
                ]
            }
        )
    else:
        results = vector_store.similarity_search(
            question,
            k=3
        )

    return "\n\n".join(
           f"Source: {doc.metadata.get('source')}\n"
    f"Content: {doc.page_content}"
    for doc in results
    )

In [ ]:

from datetime import date
from langchain.tools import tool

@tool
def create_itinerary(
    destination: str,
    start_date: date,
    end_date: date,
    question: str,
    source: str = None
):
    """
    Create an itinerary by retrieving destination facts
    from the travel knowledge base.
    """

    # Step 1: Retrieve destination facts
    search_facts = search_knowledge(question, source)

    # Step 2: Prepare prompt
    prompt = f"""
    You are a professional travel planner.

    Create a travel itinerary using the information below.

    Destination: {destination}
    Start Date: {start_date}
    End Date: {end_date}

    Customer Requirements:
    {question}

    Retrieved Destination Facts:
    {search_facts}

    Instructions:
    1. Create a day-wise itinerary.
    2. Divide each day into morning, afternoon and evening.
    3. Include nearby attractions together.
    4. Include meal and rest breaks.
    5. Include travel time and activity costs only
       when supported by the retrieved information.
    6. Do not invent opening hours or ticket prices.
    7. Suggest alternatives if weather information is provided.
    8. Mention when information is unavailable.

    Words should not be more than 60 words.
    Output format should be in format:destination\n, start_date and end_date , total_days
    day wise trip: food item for each- [morning, afternoon and evening]
    places to visit each day with travel time cost
    in the end mention total cost of trip and per person budget and mention thank you
    for choosing our plan in gentle and politr tone.
    end with beautiful short catchy quote
    """

    # Step 3: Generate itinerary
    response = llm.invoke(prompt)

    return response.content


In [ ]:
from datetime import date
from langchain.tools import tool
@tool
def ask_from_support(question:str,source:str=None):
  '''This is created to ans the query of the users regarding the travel policies of company from knowledge base'''
  search_facts = search_knowledge(question, source)
  prompt=f'''you are working as customer_assistant in travel agency company.
  you will get queries from users regarding the travel policies and you need to
  answer based on {search_facts}. Answer must be based on facts provided and
  do not provide any false information. Answers should not be more than 50 words.
  output format must in form question:{question}\n answer:{search_facts}
  and if you do not know the ans say in polite way that you do not know.
  '''

  # Step 3: Generate itinerary
  response = llm.invoke(prompt)

  return response.content

In [ ]:
support=ask_from_support.invoke(
    {
        "question":"Explain me about cancellation policy?",
    }
)
print(support)

question:Explain me about cancellation policy?  
answer:I’m sorry, but I don’t have information on the cancellation policy.


In [ ]:
itinerary = create_itinerary.invoke({
    "destination": "Jaipur",
    "start_date": date(2026, 12, 10),
    "end_date": date(2026, 12, 13),
    "question": "Include forts and local food"
})

print(itinerary)

Jaipur, 2026-12-10 to 2026-12-13, 4 days  

Day1: Kachori (morning), Dal Baati (afternoon), Lassi (evening). Amber Fort (≈30 min, cost N/A).  

Day2: Pyaaz kachori (morning), Gatte Ki Sabzi (afternoon), Jalebi (evening). Nahargarh Fort (≈20 min, cost N/A).  

Day3: Mirchi Bada (morning), Laal Maas (afternoon), Kulfi (evening). Jaigarh Fort (≈25 min, cost N/A).  

Day4: Samosa (morning), Ker Sangri (afternoon), Ghevar (evening). City Palace (≈15 min, cost N/A).  

Total cost ≈ travel + meals; per person budget ≈ ₹2,500. Thank you for choosing our plan. “Travel lights the soul.”


In [ ]:
agent = create_agent(
    model=llm,
    tools=[ask_from_support,create_itinerary,calculate_trip_budget,search_flights,search_hotels],
    system_prompt="""You are working for Travel agent company, an e-commerce company.
    you will be asked search of hotels, flights, budget of trips,creating the itineary plans ans
    queries related to policies. So, you need ans only based on facts provided and no false information.
   ans should be in polite tone and if you do not know ans mentioned that you do not
    know in humble way. If you do not have flight id, hotel id try getting them both based on destination provided.
   """,
)


In [ ]:
questions=['Calculate the budget for 3 adults staying for 4 nights, with ₹800 daily food expenses per person, ₹2,500 transport, and ₹4,000 activities for goa','Calculate the total budget for 2 adults for 3 nights. Food expenses are ₹1,000 per person per day, local transport is ₹2,000, and activities cost ₹3,000. for goa and create the itinerary plan from 23 nov to 26 nov']
for q in questions:
   response = agent.invoke({"messages": [{"role": "user", "content": q}]})
   for message in response["messages"]:
      message.pretty_print()

================================ Human Message =================================

Calculate the budget for 3 adults staying for 4 nights, with ₹800 daily food expenses per person, ₹2,500 transport, and ₹4,000 activities for goa
================================== Ai Message ==================================
Tool Calls:
  calculate_trip_budget (fc_d07134f8-7bba-404b-8d59-9e9d6524e417)
 Call ID: fc_d07134f8-7bba-404b-8d59-9e9d6524e417
  Args:
    activities: 4000
    adults: 3
    destination: Goa
    food_per_person_per_day: 800
    local_transport: 2500
    nights: 4
================================= Tool Message =================================
Name: calculate_trip_budget

{"flight_cost": 20400.0, "hotel_cost": 20800, "food_cost": 12000.0, "total_budget": 59700.0, "per_person_budget": 19900.0}
================================== Ai Message ==================================

Sure! Here’s a quick budget estimate for your 4‑night trip to Goa for **3 adults**:

| Item | Cost (₹) |
|-----